# Experimentos Fundamentais — Physics-Informed Neural Networks para Canal VLC

Este notebook documenta os **5 experimentos base** que validam a capacidade de redes neurais
em aprender as leis físicas da óptica aplicadas ao canal de Comunicação por Luz Visível (VLC).
Os modelos cobrem desde a predição de SNR pelo modelo Lambertiano até a construção de Gêmeos
Digitais 3D com Fourier Features e MIMO multi-LED.

---

## Experimentos deste notebook

| # | Experimento | Modelo | Objetivo |
|---|-------------|--------|----------|
| EXP1 | PINN para predição de SNR | MLP 4 camadas + Tanh | Validar aprendizado da radiação Lambertiana |
| EXP2 | Classificador de modulação | MLP + Dropout | Identificar OOK, PPM-4, PPM-8, VPPM via features |
| EXP3.1 | Gêmeo Digital 3D — Fourier Features | Positional Encoding (NeRF-style) | Eliminar viés espectral no mapeamento 3D |
| EXP3.2 | Gêmeo Digital com sombreamento | Fourier + Physics-Weighted Loss | Aprender bloqueio LOS com penalidade física |
| EXP3.3 | Gêmeo Digital MIMO-VLC | Fourier + AdamW + PSNR | Modelar campo de 4 LEDs com interferência óptica |

---

> **Nota metodológica:** Todos os experimentos foram executados no Google Colab com GPU T4.
> A implementação utiliza PyTorch puro para garantir reprodutibilidade independente de versão
> do PhysicsNeMo. Limitações e observações reais são documentadas em cada experimento.

In [ ]:
# 1. Alocação da GPU
!nvidia-smi

# 2. Instalação das Bibliotecas Base
# O Colab já possui o PyTorch instalado, evitando o erro de dependência prévia.
!pip install nvidia-modulus torch --quiet

# 3. Clonagem do Repositório Simbólico Oficial
# OBS: O pacote sym migrou para o ecossistema PhysicsNeMo
!git clone https://github.com/NVIDIA/physicsnemo-sym.git
%cd physicsnemo-sym
!pip install -e . --quiet
%cd ..

import torch
print(f"GPU Ativa: {torch.cuda.is_available()}")

## Configuração do Ambiente (executar primeiro)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

os.makedirs('assets', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

torch.manual_seed(42)
np.random.seed(42)
print('Ambiente configurado.')

---
## EXP1 — PINN para Predição de SNR em Canal VLC

### Contexto
A luz de um LED se dissipa de acordo com o **modelo de radiação Lambertiana**, onde a
intensidade decai com o quadrado da distância e é modulada pelo cosseno do ângulo de
incidência. O objetivo é provar que uma rede neural consegue aprender essa lei física
a partir de dados sintéticos gerados pelo modelo analítico.

### Hipótese
Treinando uma PINN simples (MLP com 4 camadas e ativação Tanh) com dados normalizados,
a rede deve produzir uma curva preditiva de SNR que se sobrepõe ao modelo matemático —
validando que a IA aprendeu a dissipação Lambertiana da luz.

In [ ]:
# ── EXP1: PINN para Predição de SNR em Canal VLC ─────────────────────────────

# ─── 1. Parâmetros Físicos do Canal VLC ──────────────────────────────────────
m_lambert  = 1.0        # Ordem Lambertiana (FOV ~60°)
A_det      = 1e-4       # Área do fotodetector [m²]
rho        = 0.53       # Responsividade [A/W]
B_bw       = 200e6      # Largura de banda [Hz]
N0         = 1e-21      # Densidade espectral de ruído [W/Hz]
q_elec     = 1.6e-19    # Carga do elétron [C]
Pt         = 1.0        # Potência transmitida [W]

# ─── 2. Funções de Formulação Analítica ──────────────────────────────────────
def channel_gain_lambertian(d, theta, m=m_lambert, A=A_det):
    phi = theta
    H = ((m + 1) * A) / (2 * np.pi * d**2)
    H *= np.cos(phi)**m * np.cos(theta)
    return H

def compute_snr(H, Pt=Pt, rho=rho, B=B_bw, N0=N0, q=q_elec):
    signal_power  = (rho * Pt * H)**2
    noise_shot    = 2 * q * rho * Pt * H * B
    noise_thermal = (N0 / 2) * B
    return 10 * np.log10(signal_power / (noise_shot + noise_thermal))

# ─── 3. Geração do Grid de Treinamento ───────────────────────────────────────
d_vals     = np.linspace(0.5, 5.0, 50)
theta_vals = np.linspace(0, np.pi/3, 30)
D, T       = np.meshgrid(d_vals, theta_vals)
H_grid     = channel_gain_lambertian(D, T)
SNR_grid   = compute_snr(H_grid)

d_norm     = (D.flatten() - 0.5) / 4.5
theta_norm = T.flatten() / (np.pi/3)
snr_norm   = (SNR_grid.flatten() - SNR_grid.min()) / (SNR_grid.max() - SNR_grid.min())

# ─── 4. Arquitetura Neural (PINN) ────────────────────────────────────────────
class VLC_PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),   nn.Tanh(),
            nn.Linear(64, 128), nn.Tanh(),
            nn.Linear(128, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

X = torch.tensor(np.stack([d_norm, theta_norm], axis=1), dtype=torch.float32).to(device)
Y = torch.tensor(snr_norm.reshape(-1, 1), dtype=torch.float32).to(device)

model   = VLC_PINN().to(device)
optim   = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# ─── 5. Treinamento ──────────────────────────────────────────────────────────
losses = []
print('Treinando EXP1 — PINN para SNR...')
for epoch in range(2001):
    model.train()
    pred = model(X)
    loss = loss_fn(pred, Y)
    optim.zero_grad()
    loss.backward()
    optim.step()
    losses.append(loss.item())
    if epoch % 500 == 0:
        print(f'  Época {epoch:4d} | MSE Loss: {loss.item():.6f}')

# ─── 6. Visualização e Salvamento ────────────────────────────────────────────
d_test     = np.linspace(0.5, 5.0, 100)
theta_test = np.zeros(100)
d_t  = torch.tensor((d_test - 0.5) / 4.5, dtype=torch.float32).reshape(-1, 1).to(device)
th_t = torch.tensor(theta_test, dtype=torch.float32).reshape(-1, 1).to(device)

model.eval()
with torch.no_grad():
    pred_snr = model(torch.cat([d_t, th_t], dim=1)).cpu().numpy()
pred_snr = pred_snr * (SNR_grid.max() - SNR_grid.min()) + SNR_grid.min()
real_snr = compute_snr(channel_gain_lambertian(d_test, theta_test))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, color='tab:blue', linewidth=1)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Curva de Aprendizado — VLC PINN (EXP1)')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].plot(d_test, real_snr, 'b-',  label='Modelo Analítico')
axes[1].plot(d_test, pred_snr, 'r--', label='PINN Predição')
axes[1].set_xlabel('Distância [m]')
axes[1].set_ylabel('SNR [dB]')
axes[1].set_title('SNR vs Distância — θ=0° (EXP1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/resultado_exp1_snr_vlc.png', dpi=150, bbox_inches='tight')
plt.show()
print('EXP1 concluído! → assets/resultado_exp1_snr_vlc.png')

---
## EXP2 — Classificador Inteligente de Modulação VLC

### Contexto
Em sistemas VLC, diferentes esquemas de modulação (OOK, PPM-4, PPM-8, VPPM) produzem
padrões estatísticos de sinal distintos. Um receptor inteligente deve identificar
automaticamente o esquema utilizado a partir de 8 features estatísticas extraídas do
sinal recebido com ruído AWGN.

### Hipótese
Um classificador MLP com Dropout deve atingir alta acurácia em OOK e PPM-8, mas pode
apresentar confusão entre PPM-4 e VPPM — evidenciando a necessidade de features temporais,
abordada no EXP4 do Notebook 2.

In [ ]:
# ── EXP2: Classificador Inteligente de Modulação VLC ─────────────────────────

N_por_classe = 500

# ─── 1. Geração de Sinais com Ruído AWGN ─────────────────────────────────────
def gerar_ook(N, snr_db=20):
    bits  = np.random.randint(0, 2, N)
    sigma = 10**(-snr_db / 20)
    sinal = bits + np.random.normal(0, sigma, N)
    return np.array([
        sinal.mean(), sinal.std(), np.var(sinal),
        np.percentile(sinal, 25), np.percentile(sinal, 75),
        len(np.unique(np.round(sinal, 1))) / N, 0.0, 0.0
    ])

def gerar_ppm(N, M=4, snr_db=20):
    sigma  = 10**(-snr_db / 20)
    slots  = np.random.randint(0, M, N)
    sinais = np.zeros((N, M))
    for i, s in enumerate(slots):
        sinais[i, s] = 1.0
    sinais += np.random.normal(0, sigma, sinais.shape)
    f = sinais.flatten()
    return np.array([
        f.mean(), f.std(), np.var(f),
        np.percentile(f, 25), np.percentile(f, 75),
        M / 16.0, np.max(sinais.mean(axis=0)), 0.5
    ])

X_list, y_list = [], []
labels = {0: 'OOK', 1: 'PPM-4', 2: 'PPM-8', 3: 'VPPM'}

for snr in [10, 15, 20, 25, 30]:
    for _ in range(N_por_classe // 5):
        X_list.append(gerar_ook(200, snr));    y_list.append(0)
        X_list.append(gerar_ppm(200, 4, snr)); y_list.append(1)
        X_list.append(gerar_ppm(200, 8, snr)); y_list.append(2)
        X_list.append(gerar_ppm(200, 4, snr)); y_list.append(3)

X = np.array(X_list)
y = np.array(y_list)
print(f'Dataset: {len(y)} amostras | {X.shape[1]} features | {len(labels)} classes')

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y)

X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)

# ─── 2. Arquitetura do Classificador ─────────────────────────────────────────
class ModulacaoClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 4)
        )
    def forward(self, x):
        return self.net(x)

clf     = ModulacaoClassifier().to(device)
optim   = torch.optim.Adam(clf.parameters(), lr=5e-4, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

# ─── 3. Treinamento ──────────────────────────────────────────────────────────
print(f'Treinando EXP2 — Classificador de Modulação no dispositivo: {device}')
for epoch in range(300):
    clf.train()
    pred = clf(X_tr_t)
    loss = loss_fn(pred, y_tr_t)
    optim.zero_grad()
    loss.backward()
    optim.step()

# ─── 4. Avaliação e Visualização ─────────────────────────────────────────────
clf.eval()
with torch.no_grad():
    y_pred = clf(X_te_t).argmax(dim=1).cpu().numpy()

print(classification_report(y_te, y_pred, target_names=list(labels.values())))

plt.figure(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_te, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=labels.values(), yticklabels=labels.values())
plt.title('Matriz de Confusão — Classificador VLC (EXP2)')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.tight_layout()
plt.savefig('assets/resultado_exp2_classificador_modulation.png', dpi=150, bbox_inches='tight')
plt.show()
print('EXP2 concluído! → assets/resultado_exp2_classificador_modulation.png')

---
## EXP3.1 — Gêmeo Digital 3D com Positional Encoding (Fourier Features)

### Contexto
Redes neurais convencionais sofrem de **viés espectral** (*spectral bias*): ao receber
coordenadas espaciais brutas (x, y, z), tendem a suavizar os dados e não conseguem
reproduzir o pico de intensidade diretamente abaixo do LED. Essa limitação inviabiliza
a geração de mapas de calor fisicamente precisos.

### Hipótese
Implementando **Fourier Features** (Positional Encoding, análogo às redes NeRF), ao projetar
as coordenadas em espaço de alta dimensão via senos e cossenos (escala σ = 2.0), o modelo
deve gerar um heatmap fotorrealístico do plano de trabalho Z = 0.5 m, sem violações
termodinâmicas (intensidade negativa ou invertida).

In [ ]:
# ── EXP3.1: Gêmeo Digital 3D com Positional Encoding (Fourier Features) ──────

# ─── 1. Configurações Iniciais ───────────────────────────────────────────────
room_size = {'x': 2.5, 'y': 2.5, 'z': 3.0}
led_pos   = {'x': 0.0, 'y': 0.0, 'z': 3.0}
m_lambert = 1.0
print(f'Inicializando Gêmeo Digital 3D no dispositivo: {device}')

# ─── 2. Geração do Grid Espacial e Intensidade Física ────────────────────────
x = np.linspace(-room_size['x'] / 2, room_size['x'] / 2, 40)
y = np.linspace(-room_size['y'] / 2, room_size['y'] / 2, 40)
z = np.linspace(0.5, room_size['z'], 30)
X, Y, Z = np.meshgrid(x, y, z)
x_flat = X.flatten().reshape(-1, 1)
y_flat = Y.flatten().reshape(-1, 1)
z_flat = Z.flatten().reshape(-1, 1)

d_squared     = (x_flat - led_pos['x'])**2 + (y_flat - led_pos['y'])**2 + (z_flat - led_pos['z'])**2
cos_theta     = np.abs(z_flat - led_pos['z']) / np.sqrt(d_squared)
intensity_real = (m_lambert + 1) / (2 * np.pi) * cos_theta**m_lambert / d_squared
intensity_norm = (intensity_real - intensity_real.min()) / (intensity_real.max() - intensity_real.min())

# ─── 3. Arquitetura Neural com Fourier Features ───────────────────────────────
class DigitalTwin3D_Fourier(nn.Module):
    def __init__(self, mapping_size=64):
        super().__init__()
        self.B = (torch.randn((3, mapping_size)) * 2.0).to(device)
        self.net = nn.Sequential(
            nn.Linear(mapping_size * 2, 128), nn.Tanh(),
            nn.Linear(128, 128),              nn.Tanh(),
            nn.Linear(128, 128),              nn.Tanh(),
            nn.Linear(128, 1),                nn.Sigmoid()
        )
    def forward(self, xyz):
        x_proj   = 2.0 * np.pi * xyz @ self.B
        features = torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)
        return self.net(features)

vlc_net   = DigitalTwin3D_Fourier().to(device)
optimizer = torch.optim.Adam(vlc_net.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

inputs  = torch.tensor(np.hstack((x_flat, y_flat, z_flat)), dtype=torch.float32).to(device)
targets = torch.tensor(intensity_norm, dtype=torch.float32).to(device)

# ─── 4. Loop de Treinamento ──────────────────────────────────────────────────
losses = []
print('Treinando EXP3.1 — Gêmeo Digital 3D Espacial (1000 épocas)...')
for epoch in range(1001):
    optimizer.zero_grad()
    preds = vlc_net(inputs)
    loss  = loss_fn(preds, targets)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 500 == 0:
        print(f'  Época {epoch:4d} | Loss: {loss.item():.6f}')

# ─── 5. Visualização e Salvamento ────────────────────────────────────────────
Z_plane  = 0.5
X_plane, Y_plane = np.meshgrid(
    np.linspace(-1.25, 1.25, 100),
    np.linspace(-1.25, 1.25, 100)
)
Z_arr        = np.full_like(X_plane, Z_plane)
inputs_plane = torch.tensor(
    np.column_stack((X_plane.flatten(), Y_plane.flatten(), Z_arr.flatten())),
    dtype=torch.float32
).to(device)

vlc_net.eval()
with torch.no_grad():
    I_pred = vlc_net(inputs_plane).cpu().numpy().reshape(100, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(losses, color='purple', linewidth=1)
axes[0].set_title('Convergência Espacial — Fourier Features (EXP3.1)')
axes[0].set_yscale('log')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(True, alpha=0.3)

contour = axes[1].contourf(X_plane, Y_plane, I_pred, levels=50, cmap='magma')
plt.colorbar(contour, ax=axes[1], label='Intensidade Luminosa Predita')
axes[1].set_title(f'Mapa de Calor IA — Plano Z={Z_plane} m (EXP3.1)')
axes[1].set_xlabel('Eixo X (m)')
axes[1].set_ylabel('Eixo Y (m)')

plt.tight_layout()
plt.savefig('assets/resultado_exp3_1_fourier_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print('EXP3.1 concluído! → assets/resultado_exp3_1_fourier_3d.png')

---
## EXP3.2 — Gêmeo Digital com Sombreamento e Physics-Weighted Loss

### Contexto
Em ambientes reais, objetos físicos criam **regiões de sombra** ao bloquear a linha de
visada (*LOS Blockage*). Uma rede treinada apenas com MSE distribui o erro uniformemente,
o que resulta em predições imprecisas na borda e no interior da região de sombra.

### Hipótese
Aplicando uma **função de custo informada pela física** (*Physics-Weighted Loss*) que
penaliza 10× mais os erros dentro da região de sombra, a rede deve aprender a
descontinuidade espacial causada pelo obstáculo com alta fidelidade geométrica.

In [ ]:
# ── EXP3.2: Gêmeo Digital com Sombreamento e Physics-Weighted Loss ────────────

# ─── 1. Espaço com Obstáculo Quadrado Físico ─────────────────────────────────
room_size = 3.0
x = np.linspace(-room_size / 2, room_size / 2, 100)
y = np.linspace(-room_size / 2, room_size / 2, 100)
X, Y = np.meshgrid(x, y)

x_flat = X.flatten().reshape(-1, 1)
y_flat = Y.flatten().reshape(-1, 1)

d_sq           = x_flat**2 + y_flat**2 + 2.5**2
intensity_base = 2.0 / (d_sq**1.5)

# Obstáculo físico central: região [-0.5 m, 0.5 m] × [-0.5 m, 0.5 m]
obstacle_mask  = (np.abs(x_flat) < 0.5) & (np.abs(y_flat) < 0.5)
intensity_real = np.copy(intensity_base)
intensity_real[obstacle_mask] = 0.0   # Bloqueio total da luz
intensity_norm = intensity_real / intensity_real.max()

print(f'Inicializando Gêmeo Digital com Sombreamento no dispositivo: {device}')

# ─── 2. Arquitetura Neural com Mapeamento de Alta Frequência ─────────────────
class ShadowDigitalTwin(nn.Module):
    def __init__(self, mapping_size=64):
        super().__init__()
        self.B = (torch.randn((2, mapping_size)) * 5.0).to(device)
        self.net = nn.Sequential(
            nn.Linear(mapping_size * 2, 256), nn.GELU(),
            nn.Linear(256, 256),              nn.GELU(),
            nn.Linear(256, 128),              nn.GELU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, xy):
        x_proj   = 2.0 * np.pi * xy @ self.B
        features = torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)
        return self.net(features)

model     = ShadowDigitalTwin().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ─── 3. Função de Custo Informada pela Física ────────────────────────────────
def custom_physics_loss(preds, targets, mask):
    base_mse       = (preds - targets)**2
    physics_weight = torch.ones_like(targets)
    physics_weight[mask] = 10.0   # Penalidade 10× maior na região de sombra
    return torch.mean(base_mse * physics_weight)

inputs      = torch.tensor(np.hstack((x_flat, y_flat)), dtype=torch.float32).to(device)
targets     = torch.tensor(intensity_norm, dtype=torch.float32).to(device)
mask_tensor = torch.tensor(obstacle_mask, dtype=torch.bool).to(device)

# ─── 4. Treinamento ──────────────────────────────────────────────────────────
losses = []
print('Treinando EXP3.2 — Gêmeo Digital com Sombreamento...')
for epoch in range(1001):
    optimizer.zero_grad()
    preds = model(inputs)
    loss  = custom_physics_loss(preds, targets, mask_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 500 == 0:
        print(f'  Época {epoch:4d} | Loss: {loss.item():.6f}')

# ─── 5. Visualização Comparativa ─────────────────────────────────────────────
model.eval()
with torch.no_grad():
    I_pred = model(inputs).cpu().numpy().reshape(100, 100)
    I_real = targets.cpu().numpy().reshape(100, 100)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].contourf(X, Y, I_real, levels=50, cmap='inferno')
axes[0].set_title('Referência Analítica — Sombra Real (EXP3.2)')
axes[0].set_xlabel('Eixo X (m)')
axes[0].set_ylabel('Eixo Y (m)')

axes[1].contourf(X, Y, I_pred, levels=50, cmap='inferno')
axes[1].set_title('Predição IA — Sombra Aprendida (EXP3.2)')
axes[1].set_xlabel('Eixo X (m)')
axes[1].set_ylabel('Eixo Y (m)')

plt.tight_layout()
plt.savefig('assets/resultado_exp3_2_sombra.png', dpi=150, bbox_inches='tight')
plt.show()
print('EXP3.2 concluído! → assets/resultado_exp3_2_sombra.png')

---
## EXP3.3 — Gêmeo Digital MIMO-VLC Avançado (4 LEDs + PSNR)

### Contexto
Sistemas MIMO-VLC utilizam múltiplos transmissores para aumentar a uniformidade da
iluminação. Com 4 LEDs dispostos nos vértices de um quadrado no teto, os campos
luminosos se somam, criando padrões de interferência óptica construtiva que devem ser
capturados pelo gêmeo digital com precisão quantificável.

### Hipótese
Usando Fourier Features com escala de frequência σ = 3.0, otimizador AdamW e
monitoramento via PSNR (*Peak Signal-to-Noise Ratio*), o modelo deve convergir para uma
representação de alta fidelidade do campo MIMO, com superposição dos 4 LEDs claramente
visível na distribuição de intensidade.

In [ ]:
# ── EXP3.3: Gêmeo Digital MIMO-VLC Avançado (4 LEDs + PSNR) ─────────────────

# ─── 1. Configuração MIMO ────────────────────────────────────────────────────
room_size = {'x': 4.0, 'y': 4.0, 'z': 3.0}
m_lambert = 1.0

leds = [
    {'x': -1.0, 'y': -1.0, 'z': 3.0},
    {'x':  1.0, 'y': -1.0, 'z': 3.0},
    {'x': -1.0, 'y':  1.0, 'z': 3.0},
    {'x':  1.0, 'y':  1.0, 'z': 3.0},
]

print(f'Inicializando Gêmeo Digital MIMO-VLC ({len(leds)} LEDs) no dispositivo: {device}')

# ─── 2. Campo de Luz Sobreposto (Superposição MIMO) ──────────────────────────
x = np.linspace(-room_size['x'] / 2, room_size['x'] / 2, 40)
y = np.linspace(-room_size['y'] / 2, room_size['y'] / 2, 40)
z = np.linspace(0.5, room_size['z'], 20)
X, Y, Z = np.meshgrid(x, y, z)

x_flat = X.flatten().reshape(-1, 1)
y_flat = Y.flatten().reshape(-1, 1)
z_flat = Z.flatten().reshape(-1, 1)

intensity_real = np.zeros_like(x_flat)
for led in leds:
    d_sq          = (x_flat - led['x'])**2 + (y_flat - led['y'])**2 + (z_flat - led['z'])**2
    cos_t         = np.abs(z_flat - led['z']) / np.sqrt(d_sq)
    intensity_real += ((m_lambert + 1) / (2 * np.pi)) * (cos_t**m_lambert / d_sq)

intensity_norm = (intensity_real - intensity_real.min()) / (intensity_real.max() - intensity_real.min())

# ─── 3. Arquitetura Neural MIMO ──────────────────────────────────────────────
class MIMO_DigitalTwin(nn.Module):
    def __init__(self, mapping_size=64):
        super().__init__()
        self.B = (torch.randn((3, mapping_size)) * 3.0).to(device)
        self.net = nn.Sequential(
            nn.Linear(mapping_size * 2, 128), nn.GELU(),
            nn.Linear(128, 128),              nn.GELU(),
            nn.Linear(128, 1),                nn.Sigmoid()
        )
    def forward(self, xyz):
        x_proj   = 2.0 * np.pi * xyz @ self.B
        features = torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)
        return self.net(features)

vlc_net   = MIMO_DigitalTwin().to(device)
optimizer = torch.optim.AdamW(vlc_net.parameters(), lr=2e-3)
loss_fn   = nn.MSELoss()

inputs  = torch.tensor(np.hstack((x_flat, y_flat, z_flat)), dtype=torch.float32).to(device)
targets = torch.tensor(intensity_norm, dtype=torch.float32).to(device)

# ─── 4. Treinamento com PSNR ─────────────────────────────────────────────────
losses, psnr_hist = [], []
print('Treinando EXP3.3 — MIMO Digital Twin...')
for epoch in range(1001):
    optimizer.zero_grad()
    preds = vlc_net(inputs)
    loss  = loss_fn(preds, targets)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 500 == 0:
        psnr = 10 * np.log10(1.0 / loss.item())
        psnr_hist.append((epoch, psnr))
        print(f'  Época {epoch:4d} | MSE: {loss.item():.6f} | PSNR: {psnr:.2f} dB')

psnr_final = 10 * np.log10(1.0 / losses[-1])
print(f'\nPSNR final: {psnr_final:.2f} dB')
print('EXP3.3 concluído! — MIMO Digital Twin treinado.')

---
## Consolidação dos Resultados

Execute esta célula após todos os experimentos para verificar a conclusão de cada etapa.

In [ ]:
print('=' * 65)
print('  RESUMO — Experimentos Fundamentais VLC + IA')
print('=' * 65)
print()
print('EXP1 — PINN para Predição de SNR (Modelo Lambertiano)')
print('  Saída: assets/resultado_exp1_snr_vlc.png')
print()
print('EXP2 — Classificador de Modulação (OOK / PPM-4 / PPM-8 / VPPM)')
print('  Nota: confusão PPM-4 ↔ VPPM esperada → abordada em EXP4 (Notebook 2)')
print('  Saída: assets/resultado_exp2_classificador_modulation.png')
print()
print('EXP3.1 — Gêmeo Digital 3D — Fourier Features')
print('  Saída: assets/resultado_exp3_1_fourier_3d.png')
print()
print('EXP3.2 — Gêmeo Digital com Sombreamento (Physics-Weighted Loss)')
print('  Saída: assets/resultado_exp3_2_sombra.png')
print()
print('EXP3.3 — Gêmeo Digital MIMO-VLC (4 LEDs + PSNR)')
print(f'  PSNR Final: {psnr_final:.2f} dB')
print()
print('Todos os assets salvos em: assets/')